# 15150SMLResearch: Colab GPU benchmark

Use a GPU runtime and run the cells in order. Upload this notebook to Colab and place `15150SMLResearch-colab.tar.gz` in `My Drive/15150SMLResearch-v2`.

The exported selection contains Ollama models only. The default five-model selection runs 31 problems × 3 specification versions × 2 repetitions × 5 models = 930 initial trials. Step 7 starts model generation. Compilation failures allow at most one repair request.

Source files, compilation and model weights use the runtime disk. Logs and results are copied to Drive every 60 seconds during logged commands and when those commands finish. A new benchmark run starts from the beginning; backups do not provide automatic resume.


## 1. Mount Drive and inspect the runtime

Change `DRIVE` if you used a different upload folder. The commands display the allocated GPU, host memory and free disk space.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, shutil, signal, subprocess, sys, tarfile, time, urllib.error, urllib.request, uuid

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/15150SMLResearch-v2')
ARCHIVE = DRIVE / '15150SMLResearch-colab.tar.gz'
ROOT = Path('/content/15150SMLResearch')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], check=True)
subprocess.run(['free', '-h'], check=True)
subprocess.run(['df', '-h', '/content'], check=True)


## 2. Extract the workspace

Run once in a fresh runtime. The archive contains the exported working files, including uncommitted changes and assignment dependencies. Preparation is rebuilt on Colab.


In [ ]:
ROOT.mkdir()
with tarfile.open(ARCHIVE) as archive:
    for member in archive.getmembers():
        destination = (ROOT.parent / member.name).resolve()
        if not destination.is_relative_to(ROOT):
            raise ValueError('Archive entry is outside the workspace: ' + member.name)
    archive.extractall(ROOT.parent, filter='data')
os.chdir(ROOT)
settings = json.loads((ROOT / 'colab.json').read_text())
config = json.loads((ROOT / 'config.json').read_text())
colab_models = settings['models']
BACKUP = DRIVE / ('session-' + time.strftime('%Y%m%d-%H%M%S') + '-' + uuid.uuid4().hex[:6])
BACKUP.mkdir(parents=True)
shutil.copyfile(ROOT / 'colab.json', BACKUP / 'colab.json')
LOGS = ROOT / 'colab-logs'
LOGS.mkdir(exist_ok=True)
print('Backup folder:', BACKUP)
print('Model order:', ' → '.join(colab_models))


## 3. Install SML/NJ and Ollama

Install the compiler and build tools, then download the selected official Ollama release. The server runs directly without systemd.


In [ ]:
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'smlnj', 'libsmlnj-smlnj', 'curl', 'zstd', 'rsync', 'make', 'perl'], check=True)
installer = Path('/content/ollama-linux-amd64.tar.zst')
subprocess.run(['curl', '-fL', '--retry', '3', '-o', str(installer), settings['ollama_release_url']], check=True)
subprocess.run(['tar', '--zstd', '-xf', str(installer), '-C', '/usr/local'], check=True)
installer.unlink()
subprocess.run(['ollama', '--version'], check=True)


## 4. Set up logging and backups

Logged commands save output locally and copy results to Drive. Interrupting a command sends SIGINT to its process group and performs a final backup. Runtime deletion can prevent that final backup.


In [ ]:
def sync_evidence():
    for relative in ('runs', 'benchmark/prepared', 'colab-logs'):
        source = ROOT / relative
        if source.exists():
            target = BACKUP / relative
            target.mkdir(parents=True, exist_ok=True)
            subprocess.run(['rsync', '-rt', '--omit-dir-times', '--no-perms',
                            '--exclude=.cm/', '--exclude=__pycache__/',
                            str(source) + '/', str(target) + '/'], check=True)

def run_logged(args, name):
    log_path = LOGS / (name + '-' + uuid.uuid4().hex[:6] + '.log')
    with log_path.open('w') as output:
        process = subprocess.Popen(args, cwd=ROOT, stdout=output, stderr=subprocess.STDOUT, start_new_session=True)
        next_sync = time.monotonic() + 60
        try:
            with log_path.open() as reader:
                while process.poll() is None:
                    print(reader.read(), end='', flush=True)
                    if time.monotonic() >= next_sync:
                        sync_evidence()
                        next_sync = time.monotonic() + 60
                    time.sleep(1)
                print(reader.read(), end='', flush=True)
        except BaseException:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGINT)
                try:
                    process.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGTERM)
                    process.wait()
            raise
        finally:
            sync_evidence()
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, args)


## 5. Start Ollama and download the selected models

The server listens on `127.0.0.1:11434`. It handles one request and keeps one model loaded at a time. Weights are stored under `/content/ollama-models`.


In [ ]:
os.environ.update(OLLAMA_HOST='127.0.0.1:11434', OLLAMA_MODELS='/content/ollama-models',
                  OLLAMA_NUM_PARALLEL='1', OLLAMA_MAX_LOADED_MODELS='1')

def api(path):
    with urllib.request.urlopen(config['ollama_url'].rstrip('/') + path,
                                timeout=config['http_timeout']) as response:
        return json.load(response)

if 'ollama_process' not in globals() or ollama_process.poll() is not None:
    with (LOGS / 'ollama-server.log').open('a') as output:
        ollama_process = subprocess.Popen(['ollama', 'serve'], stdout=output,
                                         stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(60):
    if ollama_process.poll() is not None:
        raise RuntimeError((LOGS / 'ollama-server.log').read_text())
    try:
        print('Ollama:', api('/api/version'))
        break
    except urllib.error.URLError:
        time.sleep(1)
else:
    raise RuntimeError('Ollama did not start; inspect colab-logs/ollama-server.log')
for label, model in colab_models.items():
    print('Downloading:', label, model['name'])
    run_logged(['ollama', 'pull', model['name']], 'pull')


## 6. Prepare the benchmark

Preparation builds the selected specification prompts and runs PBT for callable student helpers. It makes no model requests. Only the exported Ollama labels are selected; hosted API entries in the configuration remain inactive.

The settings and specification versions come from the packaged project. Preparation writes a plan used by the next step.


In [ ]:
model_args = [argument for label in colab_models for argument in ('--model', label)]
run_logged([sys.executable, '-u', 'benchmark.py', 'prepare', *model_args], 'prepare')
plan = json.loads((ROOT / 'benchmark/prepared/plan.json').read_text())
print('Model order:', ' → '.join(plan['models']))
print('PBT settings:', {key: config[key] for key in ('seed', 'valid_target', 'max_attempts')})
print('Initial trials:', len(plan['grid']) * len(plan['models']) * len(plan['generation_seeds']))


## 7. Run the benchmark

This cell starts model generation. Each selected model completes all problems for each specification version and repetition. Generated-predicate compilation errors permit one repair; PBT and semantic test failures do not trigger repair. Translation tests, refusal status and student program PBT are recorded separately.

Use the cell's stop button to interrupt and allow the final backup to finish. Running this cell again starts a new run.


In [ ]:
run_logged([sys.executable, '-u', 'benchmark.py', 'run'], 'benchmark')
completed = sorted((ROOT / 'runs').glob('benchmark-*/run.json'), key=lambda path: path.stat().st_mtime)[-1]
run_record = json.loads(completed.read_text())
summary = json.loads((completed.parent / 'summary.json').read_text())
print('Complete:', run_record['complete'])
print('Local results:', completed.parent)
print('Drive results:', BACKUP / 'runs' / completed.parent.name)
print(json.dumps(summary['models'], ensure_ascii=False, indent=2))


## 8. Unload models, stop Ollama and save results

Stop the benchmark cell first. This cell unloads models, terminates the server started by the notebook, and performs a final backup. Then choose **Runtime → Disconnect and delete runtime** to release the Colab allocation.


In [ ]:
try:
    try:
        for model in colab_models.values():
            subprocess.run(['ollama', 'stop', model['name']], check=True)
    finally:
        if 'ollama_process' in globals() and ollama_process.poll() is None:
            ollama_process.terminate()
            try:
                ollama_process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                ollama_process.kill()
                ollama_process.wait()
finally:
    sync_evidence()
    print('Backup folder:', BACKUP)


Results are under `My Drive/15150SMLResearch-v2/session-*/runs/benchmark-*/`. Read `run.json` for completion, then `summary.json` and `results.csv`. Each trial retains its result and response checkpoint under `<model>/<version>/<case>/<repetition>/`.

Preparation sessions, prompts and logs are also backed up. If the runtime is reclaimed abruptly, final summaries may be missing or stale, and changes since the last backup may be lost.
